# Semana 7 – Convolución de Matrices en Imágenes con Padding y Stride
**CADI Deep Learning | Universidad de Cundinamarca**

Este notebook implementa la operación de convolución **manualmente con NumPy**, la aplica sobre una imagen real, y compara el efecto de variar **padding** y **stride** manteniendo el mismo kernel. El objetivo es comprender operacionalmente cómo se generan mapas de características y cómo estas decisiones modifican las dimensiones y la información preservada.

---
## 1. Importaciones y configuración

Se importan únicamente las librerías necesarias. No se usa Keras/TF para la convolución — la implementamos desde cero con NumPy para demostrar comprensión operacional.

In [ ]:
import numpy as np                          # Álgebra lineal y operaciones matriciales
import matplotlib.pyplot as plt             # Visualización de imágenes y mapas de características
from skimage import data, color             # Imágenes de prueba estándar (incluidas en Colab)

np.random.seed(7)                           # Semilla para reproducibilidad

---
## 2. Implementación manual de la convolución 2D

### ¿Qué es la convolución?
La convolución es una operación que desliza un **kernel** (filtro pequeño) sobre una matriz de entrada, calculando en cada posición el **producto punto** entre el kernel y el parche de imagen que cubre. El resultado acumula en un **mapa de características** (*feature map*) que resalta patrones locales: bordes, texturas, gradientes, etc.

La fórmula para cada posición `(i, j)` del mapa de salida es:

$$O[i,j] = \sum_{m=0}^{k_h-1} \sum_{n=0}^{k_w-1} I[i \cdot s + m,\ j \cdot s + n] \cdot K[m, n]$$

donde `s` es el stride y `K` es el kernel.

In [ ]:
def convolve2d(image, kernel, stride=1, padding=0):
    """
    Convolución 2D implementada manualmente.

    Parámetros:
        image   : np.ndarray 2D — imagen de entrada (escala de grises)
        kernel  : np.ndarray 2D — filtro a aplicar
        stride  : int — salto del filtro en cada paso (default=1)
        padding : int — cantidad de ceros a añadir alrededor (default=0)

    Retorna:
        output  : np.ndarray 2D — mapa de características resultante
    """
    # --- Dimensiones del kernel ---
    kh, kw = kernel.shape          # Altura y ancho del kernel

    # --- Aplicar padding con zeros alrededor de la imagen ---
    # np.pad añade `padding` filas/columnas de ceros en cada borde
    if padding > 0:
        image = np.pad(image,
                       pad_width=((padding, padding), (padding, padding)),
                       mode='constant',
                       constant_values=0)

    # --- Dimensiones de la imagen (posiblemente con padding) ---
    ih, iw = image.shape

    # --- Calcular dimensiones del mapa de salida ---
    # Fórmula: out = floor((entrada - kernel) / stride) + 1
    oh = (ih - kh) // stride + 1  # Filas del mapa de salida
    ow = (iw - kw) // stride + 1  # Columnas del mapa de salida

    # --- Inicializar mapa de salida con ceros ---
    output = np.zeros((oh, ow), dtype=np.float64)

    # --- Doble bucle: desplazar el kernel sobre la imagen ---
    for i in range(oh):            # Recorre filas de la salida
        for j in range(ow):        # Recorre columnas de la salida
            # Extraer el parche de imagen que cubre el kernel
            patch = image[i*stride : i*stride+kh,
                          j*stride : j*stride+kw]
            # Producto elemento a elemento + suma escalar (producto punto)
            output[i, j] = np.sum(patch * kernel)

    return output

print("✔ Función convolve2d definida correctamente.")

---
## 3. Verificación sobre matriz pequeña

Antes de aplicar la convolución a una imagen real, verificamos que la función produce el resultado esperado sobre una matriz 4×4 controlada. Esto permite detectar errores de indexación o cálculo de forma inmediata.

In [ ]:
# Matriz de entrada 4×4 con valores conocidos
test_matrix = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
], dtype=np.float64)

# Kernel detector de bordes verticales (Sobel simplificado)
# Responde con valores altos donde hay cambio brusco de izquierda a derecha
edge_kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float64)

# --- Caso 1: sin padding, stride=1 → salida 2×2 ---
result_base = convolve2d(test_matrix, edge_kernel, stride=1, padding=0)
print("Entrada 4×4, kernel 3×3, stride=1, padding=0")
print(f"  Dimensión salida: {result_base.shape}  (esperado: 2×2)")
print(f"  Mapa de características:\n{result_base}\n")

# --- Caso 2: padding=1, stride=1 → salida 4×4 (mismo tamaño que entrada) ---
result_pad = convolve2d(test_matrix, edge_kernel, stride=1, padding=1)
print("Entrada 4×4, kernel 3×3, stride=1, padding=1")
print(f"  Dimensión salida: {result_pad.shape}  (esperado: 4×4)")
print(f"  Mapa de características:\n{result_pad}\n")

# --- Caso 3: sin padding, stride=2 → salida 1×1 ---
result_stride = convolve2d(test_matrix, edge_kernel, stride=2, padding=0)
print("Entrada 4×4, kernel 3×3, stride=2, padding=0")
print(f"  Dimensión salida: {result_stride.shape}  (esperado: 1×1)")
print(f"  Mapa de características:\n{result_stride}")

---
## 4. Carga y preprocesamiento de imagen real

Usamos la imagen estándar `camera` de scikit-image (512×512, escala de grises), disponible sin descarga adicional en Colab. Normalizamos a [0, 1] para estabilidad numérica.

In [ ]:
# Cargar imagen estándar en escala de grises (512×512 píxeles)
img = data.camera()                         # Imagen 'cameraman', incluida en skimage
img = img.astype(np.float64) / 255.0        # Normalizar a rango [0, 1]

print(f"Dimensiones de la imagen: {img.shape}")
print(f"Rango de valores: [{img.min():.3f}, {img.max():.3f}]")

plt.figure(figsize=(4, 4))
plt.imshow(img, cmap='gray')
plt.title("Imagen original (512×512)")
plt.axis('off')
plt.tight_layout()
plt.show()

---
## 5. Definición del kernel base

Se define **un único kernel Sobel horizontal** que detecta bordes verticales. Este kernel permanece **fijo en todas las comparaciones** siguientes — solo cambia padding o stride, no el filtro.

In [ ]:
# Kernel Sobel horizontal: detecta bordes (gradiente vertical de intensidad)
# Columna izquierda negativa, derecha positiva → resalta transiciones horizontales
kernel = np.array([
    [-1,  0,  1],
    [-2,  0,  2],
    [-1,  0,  1]
], dtype=np.float64)

print("Kernel Sobel (3×3):")
print(kernel)
print(f"\nSuma del kernel: {kernel.sum()} (= 0 → no cambia el brillo global)")

---
## 6. Comparación: efecto del Padding

**Variable que cambia:** padding (0 vs 1).  
**Variables fijas:** mismo kernel, stride=1.

| Configuración | padding=0 (válida) | padding=1 (same) |
|---|---|---|
| Tamaño entrada | 512×512 | 512×512 |
| Tamaño salida esperado | 510×510 | 512×512 |
| Información en bordes | Perdida | Preservada |

Con `padding=0` el mapa de salida se **reduce** en `(k-1)` píxeles por lado. Con `padding=1` la salida mantiene las mismas dimensiones que la entrada (*same padding*).

In [ ]:
# --- Convolución SIN padding (valid) ---
# La salida se reduce: (512-3)//1 + 1 = 510 en cada dimensión
fmap_no_pad = convolve2d(img, kernel, stride=1, padding=0)

# --- Convolución CON padding=1 (same) ---
# Se añade 1 fila/columna de ceros alrededor → salida conserva 512×512
fmap_pad = convolve2d(img, kernel, stride=1, padding=1)

print(f"Sin padding  → salida: {fmap_no_pad.shape}")
print(f"Con padding=1 → salida: {fmap_pad.shape}")

# --- Visualización comparativa ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img, cmap='gray')
axes[0].set_title("Original\n512×512")
axes[0].axis('off')

# np.abs para ver magnitud del gradiente; clip para estabilidad visual
axes[1].imshow(np.abs(fmap_no_pad), cmap='gray')
axes[1].set_title(f"padding=0 (valid)\n{fmap_no_pad.shape[0]}×{fmap_no_pad.shape[1]}")
axes[1].axis('off')

axes[2].imshow(np.abs(fmap_pad), cmap='gray')
axes[2].set_title(f"padding=1 (same)\n{fmap_pad.shape[0]}×{fmap_pad.shape[1]}")
axes[2].axis('off')

plt.suptitle("Efecto del Padding — mismo kernel, stride=1", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Comparación: efecto del Stride

**Variable que cambia:** stride (1, 2, 4).  
**Variables fijas:** mismo kernel, padding=1 (para hacer la comparación de tamaños más clara).

| Stride | Tamaño salida | Reducción respecto original |
|---|---|---|
| 1 | 512×512 | 1× |
| 2 | 256×256 | 4× |
| 4 | 128×128 | 16× |

Un stride mayor hace que el filtro **salte más posiciones** en cada paso, generando un mapa más pequeño que captura características de mayor escala pero pierde detalle espacial fino.

In [ ]:
# Aplicar la convolución con distintos strides (padding=1 en todos para comparar)
fmap_s1 = convolve2d(img, kernel, stride=1, padding=1)   # Salida: 512×512
fmap_s2 = convolve2d(img, kernel, stride=2, padding=1)   # Salida: 256×256
fmap_s4 = convolve2d(img, kernel, stride=4, padding=1)   # Salida: 128×128

print(f"stride=1 → salida: {fmap_s1.shape}")
print(f"stride=2 → salida: {fmap_s2.shape}")
print(f"stride=4 → salida: {fmap_s4.shape}")

# --- Visualización comparativa ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, fmap, s in zip(axes, [fmap_s1, fmap_s2, fmap_s4], [1, 2, 4]):
    ax.imshow(np.abs(fmap), cmap='gray')
    ax.set_title(f"stride={s}\n{fmap.shape[0]}×{fmap.shape[1]} px")
    ax.axis('off')

plt.suptitle("Efecto del Stride — mismo kernel, padding=1", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Análisis cuantitativo

Además de la inspección visual, comparamos métricas numéricas de los mapas de características para fundamentar las conclusiones con evidencia objetiva.

In [ ]:
print("="*55)
print(f"{'Configuración':<25} {'Shape':>10} {'Media abs':>10} {'Std':>8}")
print("-"*55)

configs = [
    ("padding=0, stride=1", fmap_no_pad),
    ("padding=1, stride=1", fmap_pad),
    ("padding=1, stride=2", fmap_s2),
    ("padding=1, stride=4", fmap_s4),
]

for label, fmap in configs:
    # Media absoluta: cuánta respuesta promedio da el filtro
    mean_abs = np.mean(np.abs(fmap))
    # Desviación estándar: qué tan variado es el mapa de características
    std = np.std(fmap)
    print(f"{label:<25} {str(fmap.shape):>10} {mean_abs:>10.4f} {std:>8.4f}")

print("="*55)
print("\nAnálisis:")
print("  • Media abs similar → el kernel detecta la misma cantidad de bordes")
print("  • Dimensiones reducen con stride mayor → submuestreo espacial")
print("  • padding=0 vs 1 mantienen estadísticas similares pero difieren en bordes")

---
## 9. Visualización detallada: zoom en bordes de la imagen

Hacemos zoom sobre la esquina superior izquierda (50×50 px) para ver con claridad la diferencia que el padding produce en los bordes del mapa de características.

In [ ]:
# Región de interés: esquina superior izquierda, 50 filas y 50 columnas
ROI = (slice(0, 50), slice(0, 50))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(img[ROI], cmap='gray')
axes[0].set_title("Original (zoom)")
axes[0].axis('off')

# padding=0 → la esquina ya no existe (el mapa es 510×510, empieza en píxel 0)
axes[1].imshow(np.abs(fmap_no_pad[ROI]), cmap='hot')
axes[1].set_title("padding=0\n(borde descartado)")
axes[1].axis('off')

# padding=1 → la esquina está preservada gracias a los ceros añadidos
axes[2].imshow(np.abs(fmap_pad[ROI]), cmap='hot')
axes[2].set_title("padding=1\n(borde preservado)")
axes[2].axis('off')

plt.suptitle("Zoom esquina superior-izquierda: impacto del padding en bordes",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Conclusiones técnicas

Basadas en los resultados observados (dimensiones de salida, visualizaciones y métricas cuantitativas):

1. **La convolución extrae características locales** mediante el producto punto entre un kernel y parches de la imagen. El kernel Sobel respondió con valores altos en regiones con cambio brusco de intensidad, confirmando que actúa como detector de bordes.

2. **El padding controla si se preservan los bordes** de la imagen. Con `padding=0` (*valid*) la salida se reduce en `(k-1)` píxeles por cada dimensión y la información de las orillas se pierde. Con `padding=1` (*same*) la salida mantiene las mismas dimensiones que la entrada, siendo esencial en redes profundas para no erosionar progresivamente la resolución espacial.

3. **El stride controla la resolución del mapa de características**. Duplicar el stride de 1 a 2 reduce cada dimensión a la mitad (512→256), y stride=4 la reduce a la cuarta parte (512→128). Esto introduce un **submuestreo** que reduce el costo computacional de capas posteriores, pero a costa de perder detalle espacial fino — comportamiento análogo al de las capas de pooling en CNNs reales.